In [0]:
# COMMAND ----------

# MAGIC %md
# MAGIC ## P2 — Quando ocorreu o pico e quão abrupta foi a escalada?
# MAGIC
# MAGIC Fonte: `gold.calendario_caso`. Com grão diário, a abruptez se mede melhor na queda do que na subida:
# MAGIC a subida do Monark é censurada pelo início da coleta e a do Arthur é mascarada pela polêmica anterior.
# MAGIC Regras: variação diária **não definida contra o primeiro dia de coleta**; razão pico/média pré-crise só com ≥ 3 dias de pré-crise.
# MAGIC Validação: 26 linhas; Σ volume = 18.696; pico 1 dia após o estopim nos dois casos.

# COMMAND ----------

p2 = spark.sql("""
  WITH s AS (
    SELECT caso, data, dias_desde_estopim, fase, volume_postagens,
           LAG(volume_postagens) OVER (PARTITION BY caso ORDER BY data)                    AS volume_dia_anterior,
           MIN(data) OVER (PARTITION BY caso)                                              AS primeiro_dia_coleta,
           MAX(CASE WHEN fase = 'pico' THEN volume_postagens END) OVER (PARTITION BY caso) AS volume_pico
    FROM gold.calendario_caso
  )
  SELECT caso, data, dias_desde_estopim, fase, volume_postagens,
         volume_postagens / volume_pico AS fracao_do_pico,
         CASE WHEN volume_dia_anterior IS NULL OR DATE_SUB(data, 1) = primeiro_dia_coleta THEN NULL
              ELSE volume_postagens / volume_dia_anterior - 1 END AS variacao_diaria
  FROM s
  ORDER BY caso, data
""").toPandas()

tot = p2.groupby("caso")["volume_postagens"].sum().to_dict()
assert len(p2) == 26 and tot == {"arthur_do_val": 13893, "monark": 4803}, f"validação P2: {len(p2)} linhas, {tot}"
assert (p2[p2.fase == "pico"].groupby("caso")["dias_desde_estopim"].min() == 1).all(), "esperava pico em dias_desde_estopim = 1"
assert p2.groupby("caso")["variacao_diaria"].apply(lambda s: s.isna().sum()).eq(2).all(), "esperava 2 variações indefinidas por caso"
print("P2 validada:", tot)
display(p2)

# COMMAND ----------

p2_resumo = spark.sql("""
  WITH c AS (
    SELECT *,
           MAX(CASE WHEN fase = 'pico' THEN dias_desde_estopim END) OVER (PARTITION BY caso) AS dia_pico,
           MAX(CASE WHEN fase = 'pico' THEN volume_postagens END)   OVER (PARTITION BY caso) AS volume_pico
    FROM gold.calendario_caso
  )
  SELECT
    caso,
    MIN(CASE WHEN fase = 'pico' THEN data END)                                              AS data_pico,
    MAX(dia_pico)                                                                           AS dias_estopim_ate_pico,
    MAX(volume_pico)                                                                        AS volume_pico,
    ROUND(MAX(volume_pico) / MAX(CASE WHEN fase = 'estopim' THEN volume_postagens END), 2) AS razao_pico_estopim,
    CASE WHEN COUNT(CASE WHEN fase = 'pre_crise' THEN 1 END) >= 3
         THEN ROUND(MAX(volume_pico) / AVG(CASE WHEN fase = 'pre_crise' THEN volume_postagens END), 2) END AS razao_pico_media_pre,
    ROUND(MAX(CASE WHEN dias_desde_estopim = dia_pico + 1 THEN volume_postagens END) / MAX(volume_pico), 2) AS fracao_dia_apos_pico,
    MIN(CASE WHEN dias_desde_estopim > dia_pico AND volume_postagens < 0.5  * volume_pico
             THEN dias_desde_estopim - dia_pico END)                                        AS dias_pico_ate_metade,
    MIN(CASE WHEN dias_desde_estopim > dia_pico AND volume_postagens < 0.25 * volume_pico
             THEN dias_desde_estopim - dia_pico END)                                        AS dias_pico_ate_quarto
  FROM c
  GROUP BY caso
  ORDER BY caso
""")
display(p2_resumo)
p2_resumo.toPandas().to_csv(f"{SAIDA}/p2_resumo.csv", index=False)

# COMMAND ----------

# figura P2 — variação dia a dia, um painel por caso
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), dpi=200, sharey=True, gridspec_kw=dict(width_ratios=[18, 8], wspace=0.08))
fig.patch.set_facecolor(SURF)

for ax, caso in zip(axes, ["arthur_do_val", "monark"]):
    eixo_limpo(ax)
    g = p2[p2.caso == caso].sort_values("dias_desde_estopim")
    v = g.dropna(subset=["variacao_diaria"])
    ax.bar(v.dias_desde_estopim, v.variacao_diaria, width=0.62, color=COR[caso], zorder=3)
    for _, r in v.iterrows():                               # borda branca fina na base: mark spec
        ax.plot([r.dias_desde_estopim - 0.31, r.dias_desde_estopim + 0.31], [0, 0], color=SURF, lw=1.2, zorder=4)
    nd = g[g.variacao_diaria.isna()]
    for _, r in nd.iterrows():
        ax.text(r.dias_desde_estopim, 0.06, "n.d.", ha="center", fontsize=7, color=MUTED)
    ax.axhline(0, color=AXIS, lw=1, zorder=2)
    ax.axvline(0, color=AXIS, lw=0.8, ls=(0, (3, 3)), zorder=1)
    pico = g[g.fase == "pico"].iloc[0]
    ax.plot([pico.dias_desde_estopim], [0.985], marker="v", ms=6, color=INK, zorder=5, clip_on=False,
            transform=ax.get_xaxis_transform(), ls="none")
    ax.text(pico.dias_desde_estopim, 1.02, "pico", ha="center", va="bottom", fontsize=7.5, color=INK2,
            transform=ax.get_xaxis_transform())
    ax.set_xticks(g.dias_desde_estopim)
    ax.set_xlim(g.dias_desde_estopim.min() - 0.7, g.dias_desde_estopim.max() + 0.7)
    ax.set_title(NOME[caso], loc="left", fontsize=10, color=INK, pad=22)
    ax.set_xlabel("dias desde o estopim", color=INK2, fontsize=9)

# anotações lidas do dado
a = p2[p2.caso == "arthur_do_val"].set_index("dias_desde_estopim")
m = p2[p2.caso == "monark"].set_index("dias_desde_estopim")
axes[0].annotate(f"28/02 (Ucrânia): +{a.loc[-4,'variacao_diaria']*100:.0f} %,\na maior variação da janela",
                 (-4, a.loc[-4, "variacao_diaria"]), xytext=(-2.6, 3.1), fontsize=7.8, color=INK2,
                 arrowprops=dict(arrowstyle="-", color=AXIS, lw=0.8))
axes[0].annotate(f"estopim → pico: +{a.loc[1,'variacao_diaria']*100:.0f} %;\ndia seguinte: {a.loc[2,'variacao_diaria']*100:.0f} %",
                 (1, a.loc[1, "variacao_diaria"]), xytext=(2.6, 1.6), fontsize=7.8, color=INK2,
                 arrowprops=dict(arrowstyle="-", color=AXIS, lw=0.8))
axes[0].annotate("14/03: corte da coleta", (10, a.loc[10, "variacao_diaria"]), xytext=(6.3, -0.85), fontsize=7.8, color=INK2,
                 arrowprops=dict(arrowstyle="-", color=AXIS, lw=0.8))
axes[1].annotate(f"dia após o pico: {m.loc[2,'variacao_diaria']*100:.0f} %", (2, m.loc[2, "variacao_diaria"]),
                 xytext=(2.3, -0.95), fontsize=7.8, color=INK2, arrowprops=dict(arrowstyle="-", color=AXIS, lw=0.8))
axes[1].annotate(f"14/02 (segunda): +{m.loc[6,'variacao_diaria']*100:.0f} %\napós o domingo",
                 (6, m.loc[6, "variacao_diaria"]), xytext=(-0.9, 2.9), fontsize=7.8, color=INK2,
                 arrowprops=dict(arrowstyle="-", color=AXIS, lw=0.8))

axes[0].set_ylim(-1.1, 3.9)
axes[0].set_yticks([-1, -0.5, 0, 0.5, 1, 2, 3])
axes[0].set_yticklabels(["−100 %", "−50 %", "0", "+50 %", "+100 %", "+200 %", "+300 %"])
axes[0].set_ylabel("variação do volume em relação ao dia anterior", color=INK2, fontsize=9)
fig.suptitle("P2 · Abruptez da escalada: variação diária do volume", x=0.02, y=0.985, ha="left", fontsize=11, color=INK)
fig.text(0.02, 0.925, "n.d. = variação não definida (primeiro dia de coleta como base) · fonte: gold.calendario_caso",
         fontsize=7.5, color=MUTED)
fig.subplots_adjust(left=0.09, right=0.99, top=0.80, bottom=0.14, wspace=0.08)
caminho = f"{SAIDA}/p2_abruptez.png"
fig.savefig(caminho, facecolor=SURF)
print("figura gravada em", caminho)
display(fig)